# LOADING DATA

In [1]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

Code directory: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code

Data paths:
  ✓ X_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote.parquet
  ✓ X_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_tomek.parquet
  ✓ X_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote_tomek.parquet
  ✓ y_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\smote\y_smote.pkl
  ✓ y_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\tomek\y_tomek.pkl
  ✓ y_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessi

# RF

In [4]:
# %%
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, f1_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ========================================
# PARTIE 1: RANDOM FOREST FROM SCRATCH 
# ========================================

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature      
        self.threshold = threshold  
        self.left = left           
        self.right = right         
        self.value = value         

class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.root = None
    
    def fit(self, X, y):
        self.n_features = X.shape[1]
        if self.max_features is None:
            self.max_features = int(np.sqrt(self.n_features))
        self.root = self._grow_tree(X, y)
    
    def _gini_impurity(self, y):
        proportions = np.bincount(y) / len(y)
        return 1 - np.sum(proportions ** 2)
    
    def _split(self, X, y, feature, threshold):
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        return X[left_mask], X[right_mask], y[left_mask], y[right_mask]
    
    def _best_split(self, X, y):
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        # Sélectionner aléatoirement un sous-ensemble de features
        features = np.random.choice(self.n_features, self.max_features, replace=False)
        
        for feature in features:
            unique_vals = np.unique(X[:, feature])
            if len(unique_vals) > 10:
                indices = np.linspace(0, len(unique_vals)-1, 10, dtype=int)
                thresholds = unique_vals[indices]
            else:
                thresholds = unique_vals
            
            for threshold in thresholds:
                X_left, X_right, y_left, y_right = self._split(X, y, feature, threshold)
                
                if len(y_left) < self.min_samples_leaf or len(y_right) < self.min_samples_leaf:
                    continue
                
                gini_parent = self._gini_impurity(y)
                n = len(y)
                n_left, n_right = len(y_left), len(y_right)
                gini_left = self._gini_impurity(y_left)
                gini_right = self._gini_impurity(y_right)
                
                gain = gini_parent - (n_left/n * gini_left + n_right/n * gini_right)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        if (depth >= self.max_depth or 
            n_labels == 1 or 
            n_samples < self.min_samples_split):
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
        
        best_feature, best_threshold = self._best_split(X, y)
        
        if best_feature is None:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
        
        X_left, X_right, y_left, y_right = self._split(X, y, best_feature, best_threshold)
        left = self._grow_tree(X_left, y_left, depth + 1)
        right = self._grow_tree(X_right, y_right, depth + 1)
        
        return Node(feature=best_feature, threshold=best_threshold, left=left, right=right)
    
    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])


class RandomForest:
    def __init__(self, n_estimators=100, max_depth=10, min_samples_split=2, 
                 min_samples_leaf=1, max_features=None, bootstrap=True):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.trees = []
    
    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]
    
    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_estimators):
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=self.max_features
            )
            
            if self.bootstrap:
                X_sample, y_sample = self._bootstrap_sample(X, y)
            else:
                X_sample, y_sample = X, y
            
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
    
    def predict(self, X):
        tree_predictions = np.array([tree.predict(X) for tree in self.trees])
        # Vote majoritaire
        predictions = []
        for i in range(X.shape[0]):
            votes = tree_predictions[:, i]
            predictions.append(Counter(votes).most_common(1)[0][0])
        return np.array(predictions)
    
    def predict_proba(self, X):
        tree_predictions = np.array([tree.predict(X) for tree in self.trees])
        n_samples = X.shape[0]
        n_classes = len(np.unique(tree_predictions))
        
        probas = np.zeros((n_samples, n_classes))
        for i in range(n_samples):
            votes = tree_predictions[:, i]
            for cls in range(n_classes):
                probas[i, cls] = np.sum(votes == cls) / self.n_estimators
        
        return probas


# ========================================
# PARTIE 2: INITIALISATION
# ========================================

print("=" * 80)
print("RANDOM FOREST - ÉVALUATION SUR TOUTES LES DONNÉES RESAMPLÉES")
print("=" * 80)

# Vérifier que les données sont chargées
if 'data' not in globals():
    print("❌ Les données ne sont pas chargées. Veuillez exécuter le code de chargement d'abord.")
    exit()

print(f"✅ Données chargées: {len(data)} datasets")

# Paramètres fixes pour toutes les exécutions
FIXED_PARAMS = {
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': None,  # sqrt par défaut
    'bootstrap': True
}

print(f"\n⚙️  PARAMÈTRES FIXES UTILISÉS:")
for param, value in FIXED_PARAMS.items():
    print(f"  {param}: {value}")

# ========================================
# PARTIE 3: ÉVALUATION SUR TOUTES LES MÉTHODES
# ========================================

# Définir les méthodes disponibles
METHODS = [
    ('SMOTE', 'X_train_smote', 'y_train_smote'),
    ('Tomek Links', 'X_train_tomek', 'y_train_tomek'),
    ('SMOTE + Tomek Links', 'X_train_smote_tomek', 'y_train_smote_tomek')
]

# Données de test communes
X_test = data['X_test'].values if 'X_test' in data else None
y_test = data['y_test'] if 'y_test' in data else None

if X_test is None or y_test is None:
    print("❌ Données de test non disponibles")
    exit()

print(f"\n📊 DONNÉES DE TEST: {X_test.shape[0]} échantillons")
print(f"🎯 Distribution des classes (Test): {Counter(y_test)}")

# Stocker tous les résultats
all_results = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ========================================
# PARTIE 4: BOUCLE SUR TOUTES LES MÉTHODES
# ========================================

for method_name, X_train_key, y_train_key in METHODS:
    print(f"\n" + "=" * 80)
    print(f"🚀 ENTRAÎNEMENT AVEC {method_name}")
    print("=" * 80)
    
    # Vérifier si les données existent
    if X_train_key not in data or y_train_key not in data:
        print(f"⚠️  Données {method_name} non disponibles, skip...")
        continue
    
    # Préparer les données
    X_train = data[X_train_key].values
    y_train = data[y_train_key]
    
    print(f"📊 Train: {X_train.shape[0]} échantillons, {X_train.shape[1]} features")
    print(f"🎯 Distribution des classes (Train): {Counter(y_train)}")
    
    # Entraînement
    start_time_method = time.time()
    
    print(f"\n🏋️‍♂️ Entraînement du Random Forest...")
    rf = RandomForest(**FIXED_PARAMS)
    rf.fit(X_train, y_train)
    
    train_time = time.time() - start_time_method
    print(f"✅ Entraînement terminé en {train_time:.2f} secondes")
    
    # Prédictions
    y_pred_train = rf.predict(X_train)
    y_pred_test = rf.predict(X_test)
    
    # Calcul des métriques (avec recall)
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_precision = precision_score(y_train, y_pred_train, average='weighted', zero_division=0)
    test_precision = precision_score(y_test, y_pred_test, average='weighted', zero_division=0)
    train_recall = recall_score(y_train, y_pred_train, average='weighted', zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, average='weighted', zero_division=0)
    train_f1 = f1_score(y_train, y_pred_train, average='weighted', zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    
    # Stocker les résultats
    method_results = {
        'method': method_name,
        'train_samples': len(y_train),
        'test_samples': len(y_test),
        'train_time': train_time,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_precision': train_precision,
        'test_precision': test_precision,
        'train_recall': train_recall,
        'test_recall': test_recall,
        'train_f1': train_f1,
        'test_f1': test_f1,
        'accuracy_gap': train_acc - test_acc,
        'precision_gap': train_precision - test_precision,
        'recall_gap': train_recall - test_recall,
        'f1_gap': train_f1 - test_f1
    }
    
    all_results.append(method_results)
    
    # Afficher les résultats détaillés
    print(f"\n📈 MÉTRIQUES {method_name}:")
    print(f"{'Métrique':<15} {'Train':<12} {'Test':<12} {'Gap':<12}")
    print("-" * 60)
    print(f"{'Accuracy':<15} {train_acc:<12.4f} {test_acc:<12.4f} {train_acc-test_acc:<12.4f}")
    print(f"{'Precision':<15} {train_precision:<12.4f} {test_precision:<12.4f} {train_precision-test_precision:<12.4f}")
    print(f"{'Recall':<15} {train_recall:<12.4f} {test_recall:<12.4f} {train_recall-test_recall:<12.4f}")
    print(f"{'F1-Score':<15} {train_f1:<12.4f} {test_f1:<12.4f} {train_f1-test_f1:<12.4f}")
    
    # Classification Report
    print(f"\n📊 CLASSIFICATION REPORT {method_name} (Test):")
    report = classification_report(y_test, y_pred_test, zero_division=0)
    print(report)
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred_test)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matrice de Confusion - {method_name}', fontsize=14, fontweight='bold')
    plt.ylabel('Vraie classe', fontsize=12)
    plt.xlabel('Classe prédite', fontsize=12)
    plt.tight_layout()
    cm_filename = f"confusion_matrix_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
    plt.savefig(cm_filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"💾 Matrice de confusion sauvegardée: {cm_filename}")

# ========================================
# PARTIE 5: COMPARAISON DES MÉTHODES
# ========================================

print("\n" + "=" * 80)
print("📊 COMPARAISON DES MÉTHODES DE RESAMPLING")
print("=" * 80)

if all_results:
    # Créer un DataFrame avec tous les résultats
    comparison_df = pd.DataFrame(all_results)
    
    # Afficher le tableau comparatif
    print("\n📋 TABLEAU COMPARATIF (Performance sur Test):")
    print("-" * 100)
    print(f"{'Méthode':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Temps (s)':<12}")
    print("-" * 100)
    
    for result in all_results:
        print(f"{result['method']:<25} "
              f"{result['test_accuracy']:<12.4f} "
              f"{result['test_precision']:<12.4f} "
              f"{result['test_recall']:<12.4f} "
              f"{result['test_f1']:<12.4f} "
              f"{result['train_time']:<12.2f}")
    
    print("-" * 100)
    
    # Analyse des gaps (overfitting)
    print("\n🔍 ANALYSE DE L'OVERFITTING (Gap = Train - Test):")
    print("-" * 100)
    print(f"{'Méthode':<25} {'Accuracy Gap':<15} {'F1 Gap':<15} {'Statut':<20}")
    print("-" * 100)
    
    for result in all_results:
        accuracy_gap = result['accuracy_gap']
        f1_gap = result['f1_gap']
        max_gap = max(abs(accuracy_gap), abs(f1_gap))
        
        if max_gap > 0.15:
            status = "❌ OVERFITTING SÉVÈRE"
        elif max_gap > 0.08:
            status = "⚠️ OVERFITTING MODÉRÉ"
        elif max_gap > 0.03:
            status = "✅ LÉGÈRE SURADAPTATION"
        else:
            status = "✅ EXCELLENT"
        
        print(f"{result['method']:<25} "
              f"{accuracy_gap:<15.4f} "
              f"{f1_gap:<15.4f} "
              f"{status:<20}")
    
    print("-" * 100)
    
    # Déterminer la meilleure méthode
    best_method_idx = np.argmax([r['test_f1'] for r in all_results])
    best_method = all_results[best_method_idx]
    
    print(f"\n🏆 MEILLEURE MÉTHODE: {best_method['method']}")
    print(f"   📊 F1-Score: {best_method['test_f1']:.4f}")
    print(f"   🎯 Accuracy: {best_method['test_accuracy']:.4f}")
    print(f"   📈 Precision: {best_method['test_precision']:.4f}")
    print(f"   🔄 Recall: {best_method['test_recall']:.4f}")
    print(f"   ⏱️  Temps d'entraînement: {best_method['train_time']:.2f} secondes")
    
    # ========================================
    # PARTIE 6: VISUALISATIONS DE COMPARAISON
    # ========================================
    
    # Graphique 1: Comparaison des métriques sur Test
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    methods = [r['method'] for r in all_results]
    test_accuracies = [r['test_accuracy'] for r in all_results]
    test_precisions = [r['test_precision'] for r in all_results]
    test_recalls = [r['test_recall'] for r in all_results]
    test_f1_scores = [r['test_f1'] for r in all_results]
    train_times = [r['train_time'] for r in all_results]
    
    # Palette de couleurs
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    # Graphique 1: Métriques principales
    x = np.arange(len(methods))
    width = 0.2
    
    bars1 = axes[0, 0].bar(x - width*1.5, test_accuracies, width, label='Accuracy', color=colors[0], edgecolor='black')
    bars2 = axes[0, 0].bar(x - width/2, test_precisions, width, label='Precision', color=colors[1], edgecolor='black')
    bars3 = axes[0, 0].bar(x + width/2, test_recalls, width, label='Recall', color=colors[2], edgecolor='black')
    bars4 = axes[0, 0].bar(x + width*1.5, test_f1_scores, width, label='F1-Score', color='purple', edgecolor='black')
    
    axes[0, 0].set_title('Comparaison des Métriques (Test)', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Méthode', fontsize=12)
    axes[0, 0].set_ylabel('Score', fontsize=12)
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(methods, rotation=0)
    axes[0, 0].set_ylim([0, 1.05])
    axes[0, 0].legend(loc='upper left', bbox_to_anchor=(1, 1))
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for bars in [bars1, bars2, bars3, bars4]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Graphique 2: Temps d'entraînement
    bars_time = axes[0, 1].bar(methods, train_times, color=colors, edgecolor='black')
    axes[0, 1].set_title('Temps d\'Entraînement', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Méthode', fontsize=12)
    axes[0, 1].set_ylabel('Temps (secondes)', fontsize=12)
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    for bar in bars_time:
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}s', ha='center', va='bottom', fontsize=11)
    
    # Graphique 3: Comparaison Train vs Test (Accuracy)
    train_accs = [r['train_accuracy'] for r in all_results]
    test_accs = [r['test_accuracy'] for r in all_results]
    
    x = np.arange(len(methods))
    bars_train = axes[1, 0].bar(x - width/2, train_accs, width, label='Train', color='#1f77b4', edgecolor='black', alpha=0.8)
    bars_test = axes[1, 0].bar(x + width/2, test_accs, width, label='Test', color='#ff7f0e', edgecolor='black', alpha=0.8)
    
    axes[1, 0].set_title('Accuracy: Train vs Test', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Méthode', fontsize=12)
    axes[1, 0].set_ylabel('Accuracy', fontsize=12)
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(methods)
    axes[1, 0].set_ylim([0, 1.05])
    axes[1, 0].legend()
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    for bars in [bars_train, bars_test]:
        for bar in bars:
            height = bar.get_height()
            axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.3f}', ha='center', va='bottom', fontsize=10)
    
    # Graphique 4: Gaps d'overfitting
    accuracy_gaps = [r['accuracy_gap'] for r in all_results]
    f1_gaps = [r['f1_gap'] for r in all_results]
    
    x = np.arange(len(methods))
    bars_acc_gap = axes[1, 1].bar(x - width/2, accuracy_gaps, width, label='Accuracy Gap', color='#d62728', edgecolor='black', alpha=0.7)
    bars_f1_gap = axes[1, 1].bar(x + width/2, f1_gaps, width, label='F1 Gap', color='#9467bd', edgecolor='black', alpha=0.7)
    
    # Lignes de référence pour l'overfitting
    axes[1, 1].axhline(y=0.03, color='green', linestyle='--', alpha=0.5, label='Seuil normal')
    axes[1, 1].axhline(y=0.08, color='orange', linestyle='--', alpha=0.5, label='Avertissement')
    axes[1, 1].axhline(y=0.15, color='red', linestyle='--', alpha=0.5, label='Overfitting')
    axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    axes[1, 1].set_title('Analyse des Gaps (Train - Test)', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Méthode', fontsize=12)
    axes[1, 1].set_ylabel('Gap', fontsize=12)
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(methods)
    axes[1, 1].legend(loc='upper left', bbox_to_anchor=(1, 1))
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    for bars in [bars_acc_gap, bars_f1_gap]:
        for bar in bars:
            height = bar.get_height()
            axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.3f}', ha='center', va='bottom' if height > 0 else 'top', fontsize=10)
    
    plt.suptitle(f'Comparaison des Méthodes de Resampling - Random Forest\n{timestamp}', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Sauvegarder le graphique de comparaison
    comparison_plot_filename = f"comparison_plot_{timestamp}.png"
    plt.savefig(comparison_plot_filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"\n💾 Graphique de comparaison sauvegardé: {comparison_plot_filename}")
    
    # Graphique 5: Radar chart pour les métriques
    fig = plt.figure(figsize=(10, 8))
    
    # Préparer les données pour le radar chart
    metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', '1/Temps']
    n_metrics = len(metrics_names)
    
    # Normaliser le temps (inversé car moins de temps est mieux)
    max_time = max(train_times)
    normalized_times = [1 - (t / max_time) for t in train_times]
    
    angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
    angles += angles[:1]  # Fermer le cercle
    
    ax = fig.add_subplot(111, polar=True)
    
    for i, result in enumerate(all_results):
        values = [
            result['test_accuracy'],
            result['test_precision'],
            result['test_recall'],
            result['test_f1'],
            normalized_times[i]  # Utiliser le temps normalisé
        ]
        values += values[:1]  # Fermer le cercle
        
        ax.plot(angles, values, 'o-', linewidth=2, label=result['method'], color=colors[i])
        ax.fill(angles, values, alpha=0.1, color=colors[i])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_names, fontsize=12)
    ax.set_ylim([0, 1])
    ax.set_title('Radar Chart des Performances', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax.grid(True)
    
    radar_filename = f"radar_chart_{timestamp}.png"
    plt.savefig(radar_filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"💾 Radar chart sauvegardé: {radar_filename}")
    
    # ========================================
    # PARTIE 7: RÉSUMÉ FINAL
    # ========================================
    
    print("\n" + "=" * 80)
    print("📋 RÉSUMÉ FINAL")
    print("=" * 80)
    
    print(f"\n📁 Fichiers générés:")
    print(f"   • Matrices de confusion: 3 fichiers PNG")
    print(f"   • Graphique de comparaison: {comparison_plot_filename}")
    print(f"   • Radar chart: {radar_filename}")
    
    print(f"\n⏱️  Timestamp: {timestamp}")
    print("=" * 80)

else:
    print("❌ Aucune donnée disponible pour l'évaluation")

print("\n✅ Exécution terminée avec succès!")

RANDOM FOREST - ÉVALUATION SUR TOUTES LES DONNÉES RESAMPLÉES
✅ Données chargées: 10 datasets

⚙️  PARAMÈTRES FIXES UTILISÉS:
  n_estimators: 100
  max_depth: 10
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: None
  bootstrap: True

📊 DONNÉES DE TEST: 3477 échantillons
🎯 Distribution des classes (Test): Counter({0: 3275, 1: 202})

🚀 ENTRAÎNEMENT AVEC SMOTE
📊 Train: 30570 échantillons, 64 features
🎯 Distribution des classes (Train): Counter({0: 15285, 1: 15285})

🏋️‍♂️ Entraînement du Random Forest...
✅ Entraînement terminé en 5315.15 secondes

📈 MÉTRIQUES SMOTE:
Métrique        Train        Test         Gap         
------------------------------------------------------------
Accuracy        0.9461       0.8896       0.0566      
Precision       0.9473       0.9447       0.0026      
Recall          0.9461       0.8896       0.0566      
F1-Score        0.9461       0.9099       0.0362      

📊 CLASSIFICATION REPORT SMOTE (Test):
              precision    recall  f1-score